In [ ]:
%load_ext watermark


In [ ]:
import os
import random

import alifedata_phyloinformatics_convert as apc
from hstrat import _auxiliary_lib as hstrat_aux
from matplotlib import pyplot as plt
import numpy as np
import iplotx as ipx
import pandas as pd
from teeplot import teeplot as tp

hstrat_aux.seed_random(5)


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get("NOTEBOOK_NAME", "2025-10-20-mls-strong-fossils__wse-async-ga")
teeplot_subdir


## Prep Data


In [ ]:
df = pd.read_parquet("https://osf.io/download/u4kzw/")


In [ ]:
df = hstrat_aux.alifestd_downsample_tips_clade_asexual(df, 512, seed=10)


In [ ]:
df = hstrat_aux.alifestd_to_working_format(df)
df["origin_time"] = df["hstrat_rank_from_t0"]
df = hstrat_aux.alifestd_mark_origin_time_delta_asexual(df)
df.fillna({"origin_time": 0}, inplace=True)
df


In [ ]:
clip = 100_000
df["extant"] = df["origin_time"] > clip
dfx = hstrat_aux.alifestd_prune_extinct_lineages_asexual(df)
dfx = hstrat_aux.alifestd_collapse_unifurcations(dfx)

tree = apc.alife_dataframe_to_dendropy_tree(
    hstrat_aux.alifestd_try_add_ancestor_list_col(
        dfx,
    ),
    setattrs=[
        "focal_trait_count",
        *[f"trait_num{i}" for i in range(64)],
    ],
    # setup_edge_lengths=True,
)


In [ ]:
with hstrat_aux.RngStateContext(6):
    tree.ladderize()
    tree.reorder(key=lambda *args: max(random.random(), 0.89))


## Example Plot


In [ ]:
with tp.teed(
    ipx.plotting.tree,
    tree,
    aspect=0.7,
    edge_color="darkgray",
    edge_linewidth=0.9,
    ladderize=True,
    layout="radial",
    layout_angular=True,
    layout_start=0.25,
    margins=0,
) as teed:
    ax = teed.axes
    plt.gcf().set_size_inches(6, 6)

    for n, (x, y) in teed.get_layout().T.items():
        x_, y_ = x * np.cos(y), x * np.sin(y)
        if n.focal_trait_count == 1.0:
            plt.plot(x_, y_, "d", markersize=8, color="#E2FFC4", alpha=0.8, zorder=0)
            plt.plot(x_, y_, "d", markersize=5, color="#6AB81C", alpha=1, zorder=2)
        if n.trait_num17 == 1.0:
            plt.plot(x_, y_, "o", markersize=5, color="#C5A3FF", alpha=0.3, zorder=1)
            plt.plot(x_, y_, "o", markersize=2.5, color="#C5A3FF", alpha=1.0, zorder=3)
        if n is tree.seed_node:
            plt.plot(x_, y_, "o", markersize=3.5, color="black", alpha=1, fillstyle="none")
            plt.plot(x_, y_, "o", markersize=5.5, color="black", alpha=1, fillstyle="none")

    plt.gca().margins(x=-0.1,y=-0.05)

    h0, = plt.plot([], [], "d", markersize=0, color="#FFFFFF", label="Mutation | ")
    h1, = plt.plot([], [], "d", markersize=8, color="#6AB81C", label="MLS")
    h2, = plt.plot([], [], "o", markersize=8, color="#C5A3FF", label="Neutral")

    plt.legend(
        handles=[h0, h1, h2],
        loc='upper center',
        bbox_to_anchor=(0.48, 0.05),
        frameon=False,
        ncol=4,
        fontsize=18,
        columnspacing=0.2,
        handletextpad=-0.2,
    )
